# Lesson 10: Generative Adversarial Networks (GANs) & Neural Style Transfer

In this notebook, you will:
1. Train a **DCGAN** (Generator + Discriminator) on MNIST digits using PyTorch.
2. Implement **Gram-Matrix Style Loss** for Neural Artistic Style Transfer.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tfms = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
loader = DataLoader(datasets.MNIST("./data", train=True, download=True, transform=tfms), batch_size=128, shuffle=True)

z_dim = 64
G = nn.Sequential(
    nn.Linear(z_dim, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 784), nn.Tanh()
).to(device)
D = nn.Sequential(
    nn.Linear(784, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 1), nn.Sigmoid()
).to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=2e-4)
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4)
bce = nn.BCELoss()

for epoch in range(2):
    for xb, _ in loader:
        bs = xb.size(0)
        real = xb.view(bs, -1).to(device)
        z = torch.randn(bs, z_dim, device=device)
        fake = G(z)

        # Train Discriminator
        d_loss = bce(D(real), torch.ones(bs, 1, device=device)) + bce(D(fake.detach()), torch.zeros(bs, 1, device=device))
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

        # Train Generator
        g_loss = bce(D(fake), torch.ones(bs, 1, device=device))
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()
    print(f"Epoch {epoch+1} | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")

In [ ]:
def gram_matrix(tensor):
    b, c, h, w = tensor.size()
    features = tensor.view(b * c, h * w)
    G_mat = torch.mm(features, features.t())
    return G_mat.div(b * c * h * w)

dummy_feat = torch.randn(1, 64, 16, 16)
print("Gram Matrix shape:", tuple(gram_matrix(dummy_feat).shape))